### Example Fixed Project

The rest of this tutorial uses pre compiled ORBIT configs that are stored as .yaml files in the '~/configs/ folder. There are load and save methods available in ORBIT for working with .yaml files. These example projects each exhibit different functionalities within ORBIT. Using these examples and combinations of them, most project configurations can be modeled. 

In [1]:
import os
import pandas as pd
from ORBIT import ProjectManager, load_config

weather = pd.read_csv("data/era5_40.0N_72.5W_1990_2020.csv", parse_dates=["datetime"])\
            .set_index("datetime")

### Load the project configuration

In [2]:
fixed_config = load_config("configs/example_fixed_project_600MW.yaml")  # Configs can be loaded with absolute or relative paths

print(type(fixed_config))                                         # They are loaded in as dictionaries.

print(f"Num turbines: {fixed_config['plant']['num_turbines']}")   # Once a configuration is loaded, different parameters can  
print(f"Turbine: {fixed_config['turbine']}")                      # be accessed using dict access.
print(f"\nSite: {fixed_config['site']}")

<class 'dict'>
Num turbines: 50
Turbine: 12MW_generic

Site: {'depth': 34, 'distance': 116, 'distance_to_landfall': 50, 'mean_windspeed': 9.17}


### Phases

This fixed project represents a generic Offshore Wind farm with 50 6MW turbines. It includes 5 design modules and 6 installation modules as seen below. This is a common set of modules to run for a fixed bottom project. This config will model the procurement and installation of monopiles, scour protection, array system, export system, offshore substation and the turbines.

In [3]:
print(f"Design phases: {fixed_config['design_phases']}")
print(f"\nInstall phases: {list(fixed_config['install_phases'].keys())}")

Design phases: ['CustomArraySystemDesign', 'ElectricalDesign', 'MonopileDesign', 'ScourProtectionDesign']

Install phases: ['ArrayCableInstallation', 'ExportCableInstallation', 'MonopileInstallation', 'OffshoreSubstationInstallation', 'ScourProtectionInstallation', 'TurbineInstallation']


### Run

This project is always being modeled with the example weather project supplied that is representative of US East Coast wind farm locations.

In [4]:
project = ProjectManager(fixed_config, weather=weather)
project.run()

ORBIT library intialized at 'C:\ORBIT_procurement_by_year\ORBIT-2023\ORBIT\library'


Missing data in columns ['bury_speed']; all values will be calculated.DeprecationWarning: C:\ORBIT_procurement_by_year\ORBIT-2023\ORBIT\ORBIT\manager.py:730
landfall dictionary will be deprecated and moved into [export_system][landfall].

### Top Level Outputs

ProjectManager offers several high level result categories:
- Installation CapEx
- System CapEx (procurement of BOS subcomponents)
- Turbine CapEx
- Soft CapEx (project management costs)
- Total CapEx
- Total installation time
- etc.

In [5]:
print(f"Installation CapEx:  {project.installation_capex/1e6:.0f} M")
print(f"System CapEx:        {project.system_capex/1e6:.0f} M")
print(f"Turbine CapEx:       {project.turbine_capex/1e6:.0f} M")
print(f"Soft CapEx:          {project.soft_capex/1e6:.0f} M")
print(f"Total CapEx:        {project.total_capex/1e6:.0f} M")

print(f"\nInstallation Time: {project.installation_time:.0f} h")

Installation CapEx:  273 M
System CapEx:        1048 M
Turbine CapEx:       1062 M
Soft CapEx:          521 M
Total CapEx:        3084 M

Installation Time: 16904 h


### CapEx Breakdown

In [6]:
# The breakdown of project costs by module is available  at 'capex_breakdown'
data = project.capex_detailed_soft_capex_breakdown_per_kw

# Convert the dictionary into a pandas DataFrame
df = pd.DataFrame(list(data.items()), columns=['Category', 'Value'])

# Add a "Total" row
total_row = pd.DataFrame([['Total', df['Value'].sum()]], columns=['Category', 'Value'])
df = pd.concat([df, total_row], ignore_index=True)

# Display the DataFrame
df.to_csv("fixed_costs.csv")
display(df)

,Category,Value
0,Array System,216.995916
1,Export System,429.571232
2,Offshore Substation,382.982715
3,Scour Protection,8.910000
4,Substructure,708.106026
5,Array System Installation,148.753818
6,Export System Installation,40.682403
7,Offshore Substation Installation,8.888424
8,Scour Protection Installation,28.157203
9,Substructure Installation,83.403832


### Installation Actions

In [7]:
df = pd.DataFrame(project.actions)    # The project simulation logs are also available for all modules
df

,cost_multiplier,agent,action,duration,cost,level,time,phase,location,phase_name,site_depth,max_waveheight,max_windspeed,transit_speed,hub_height,per_trip
0,0.5,Array Cable Installation Vessel,Mobilize,72.000000,3.513930e+05,ACTION,0.000000,ArrayCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.5,Export Cable Installation Vessel,Mobilize,72.000000,3.513930e+05,ACTION,0.000000,ExportCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.5,Export Cable Burial Vessel,Mobilize,72.000000,3.513930e+05,ACTION,0.000000,ExportCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Onshore Construction,Onshore Construction,0.000000,4.446058e+06,ACTION,0.000000,ExportCableInstallation,Landfall,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.5,Heavy Lift Vessel,Mobilize,72.000000,9.100755e+05,ACTION,0.000000,OffshoreSubstationInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3139,NaN,Array Cable Installation Vessel,Lay/Bury Cable,122.577584,1.196470e+06,ACTION,6771.816385,ArrayCableInstallation,NaN,ArrayCableInstallation,NaN,2.0,25.0,11.5,NaN,NaN
3140,NaN,Array Cable Installation Vessel,Prepare Cable,1.000000,9.760917e+03,ACTION,6772.816385,ArrayCableInstallation,NaN,ArrayCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN
3141,NaN,Array Cable Installation Vessel,Pull In Cable,5.500000,5.368504e+04,ACTION,6778.316385,ArrayCableInstallation,NaN,ArrayCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN
3142,NaN,Array Cable Installation Vessel,Terminate Cable,5.500000,5.368504e+04,ACTION,6783.816385,ArrayCableInstallation,NaN,ArrayCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# These logs can be sorted by phase by using DataFrame operations

turbine_install = df.loc[df['phase']=="TurbineInstallation"]
turbine_install

,cost_multiplier,agent,action,duration,cost,level,time,phase,location,phase_name,site_depth,max_waveheight,max_windspeed,transit_speed,hub_height,per_trip
385,1.0,WTIV,Mobilize,168.000000,2.719780e+06,ACTION,1130.315329,TurbineInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
386,NaN,WTIV,Fasten Tower Section,4.000000,6.475667e+04,ACTION,1134.315329,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,132.0,NaN
388,NaN,WTIV,Fasten Tower Section,4.000000,6.475667e+04,ACTION,1138.315329,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,132.0,NaN
389,NaN,WTIV,Fasten Nacelle,4.000000,6.475667e+04,ACTION,1142.315329,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,132.0,NaN
390,NaN,WTIV,Fasten Blade,1.500000,2.428375e+04,ACTION,1143.815329,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,132.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3041,NaN,WTIV,Attach Blade,3.500000,5.666208e+04,ACTION,5437.835329,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,132.0,NaN
3042,NaN,WTIV,Release Blade,1.000000,1.618917e+04,ACTION,5438.835329,TurbineInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3044,NaN,WTIV,Lift Blade,1.320000,2.136970e+04,ACTION,5440.155329,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,132.0,NaN
3046,NaN,WTIV,Attach Blade,3.500000,5.666208e+04,ACTION,5443.655329,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,132.0,NaN


In [9]:
# Operations can also be grouped to see a total amount of time spend on each operation

turbine_install.groupby(["action"]).sum()['duration']

action
Attach Blade             525.000000
Attach Nacelle           300.000000
Attach Tower Section     600.000000
Delay                    293.000000
Fasten Blade             225.000000
Fasten Nacelle           200.000000
Fasten Tower Section     400.000000
Jackdown                  19.666667
Jackup                    19.666667
Lift Blade               198.000000
Lift Nacelle              66.000000
Lift Tower Section        99.000000
Mobilize                 168.000000
Position Onsite          100.000000
Reequip                  100.000000
Release Blade            150.000000
Release Nacelle          150.000000
Release Tower Section    300.000000
Transit                  568.400000
Name: duration, dtype: float64